# 83 — Simulate10Next: Conqueror & Supplier Tests

Test notebook for `StrategyPipeline` from `82-Simulate10Next_Conqueror_Supplier.py`.
Each test exposes `df_s`, `pa`, `safe`, and `action` for interactive inspection.

In [ ]:
%run 82-Simulate10Next_Conqueror_Supplier.py

In [ ]:
import copy, math, random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import kaggle_environments as ke

_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}


class Obs:
    def __init__(self, planets, initial_planets=None, fleets=None,
                 next_fleet_id=100, comets=None, comet_planet_ids=None,
                 angular_velocity=0.0):
        self.planets          = [list(p) for p in planets]
        self.initial_planets  = [list(p) for p in (initial_planets if initial_planets is not None else planets)]
        self.fleets           = [list(f) for f in (fleets or [])]
        self.next_fleet_id    = next_fleet_id
        self.comets           = comets or []
        self.comet_planet_ids = comet_planet_ids or []
        self.angular_velocity = angular_velocity


def simulate_with_action(obs, action0, n_steps, current_step=0):
    snapshots = []
    for i, step in enumerate(range(current_step, current_step + n_steps)):
        snapshots.append({
            'step':    step,
            'planets': [p[:] for p in obs.planets],
            'fleets':  [f[:] for f in obs.fleets],
        })
        interpreter(obs, [action0 if i == 0 else [], []], step)
    return snapshots


def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')

    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, "+"+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)
            ax.text(x + 1.5, y + 1.5, str(ships), color=c, fontsize=5, zorder=6)
        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

## Test 01 — 1 Supplier, 1 Conqueror, 1 Enemy

Planet 1 (x=30) should attack planet 2 (x=70). Planet 0 (x=20) stays as supplier.

In [ ]:
obs01 = Obs(
    planets=[
        [0, 0, 20.0, 20.0, 1 + math.log(3), 50, 3],  # Supplier
        [1, 0, 30.0, 20.0, 1 + math.log(3), 50, 3],  # Conqueror
        [2, 1, 70.0, 20.0, 1.0,              1,  1],  # Enemy
    ],
    angular_velocity=0.0,
)
df_s01, pd01 = StrategyPipeline._01_get_obs_dataframe(obs01, step=0, num_agents=2)
df_s01

In [ ]:
pa01 = StrategyPipeline._02_get_all_opportunities(df_s01, pd01, player_id=0)
pa01

In [ ]:
safe01 = StrategyPipeline._03_filter_collision(pa01)
safe01

In [ ]:
action01 = StrategyPipeline._04_score_and_decide(safe01, player_id=0)
print("Action:", action01)
snaps01 = simulate_with_action(copy.deepcopy(obs01), action01, 20)
make_animation(snaps01, title='Test 01 — 1 Supplier, 1 Conqueror, 1 Enemy', interval=200)